In [18]:
!pip install torch torchvision torchaudio --quiet
!pip install torch-geometric --quiet
!pip install pandas numpy tables matplotlib --quiet

In [19]:

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader  # Non-deprecated import
from torch_geometric.nn import TransformerConv, BatchNorm
from torch_geometric.transforms import AddRandomWalkPE
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import os
import zipfile

# Set device (Colab GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [27]:
zip_path = '/content/METR-LA.zip'
data_dir = '/content/METR-LA'
if not os.path.exists(data_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)

In [31]:
def load_metr_la_data(data_dir='METR-LA'):
    adj_mat = np.load(os.path.join(data_dir, 'adj_mat.npy'))  # Shape: [207, 207]
    node_values = np.load(os.path.join(data_dir, 'node_values.npy'))  # Shape: [34272, 207, 2]

    # Handle 3D node_values (select speed feature and transpose)
    print(f"Original node_values shape: {node_values.shape}")
    if node_values.ndim == 3:
        node_values = node_values[:, :, 0]  # Shape: [34272, 207]
        node_values = node_values.T  # Shape: [207, 34272]
    elif node_values.ndim != 2:
        raise ValueError(f"Expected 2D or 3D node_values, got shape {node_values.shape}")

    print(f"Processed node_values shape: {node_values.shape}")

    # Normalize node values
    scaler = MinMaxScaler()
    node_values = scaler.fit_transform(node_values.T).T  # Shape: [207, 34272]

    # Convert adjacency matrix to edge_index and edge_weight
    edge_index = np.where(adj_mat > 0)  # Tuple of NumPy arrays
    edge_index = torch.tensor(edge_index, dtype=torch.long)  # Create CPU tensor
    edge_weight = torch.tensor(adj_mat[edge_index[0], edge_index[1]], dtype=torch.float)  # Create CPU tensor
    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)
    print(f"edge_index device: {edge_index.device}, shape: {edge_index.shape}")
    print(f"edge_weight device: {edge_weight.device}, shape: {edge_weight.shape}")

    return node_values, edge_index, edge_weight, scaler

In [29]:
def create_data_samples(node_values, in_timestep=12, out_timestep=12):
    features, labels = [], []
    for i in range(node_values.shape[1] - in_timestep - out_timestep + 1):
        features.append(node_values[:, i:i + in_timestep])
        labels.append(node_values[:, i + in_timestep:i + in_timestep + out_timestep])
    features = torch.tensor(features, dtype=torch.float).permute(0, 2, 1)
    labels = torch.tensor(labels, dtype=torch.float).permute(0, 2, 1)
    return features, labels

In [30]:
class GraphTransformer(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, pe_dim=20, heads=4):
        super().__init__()
        self.node_emb = nn.Linear(in_channels, hidden_channels - pe_dim)
        self.pe_lin = nn.Linear(pe_dim, pe_dim)
        self.convs = nn.ModuleList([
            TransformerConv(hidden_channels, hidden_channels, heads=heads, concat=False, dropout=0.1)
            for _ in range(num_layers)
        ])
        self.norm = BatchNorm(hidden_channels)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, out_channels)
        )

    def forward(self, x, edge_index, pe, batch):
        x = self.node_emb(x)
        pe = self.pe_lin(pe)
        x = torch.cat([x, pe], dim=-1)
        for conv in self.convs:
            x = conv(x, edge_index)
            x = self.norm(x)
            x = torch.relu(x)
        x = self.mlp(x)
        batch_size = batch.max().item() + 1
        num_nodes = x.size(0) // batch_size
        return x.view(batch_size, num_nodes, -1)


In [32]:
def compute_metrics(pred, target):
    mae = torch.mean(torch.abs(pred - target)).item()
    rmse = torch.sqrt(torch.mean((pred - target) ** 2)).item()
    mape = torch.mean(torch.abs((pred - target) / (target + 1e-8))) * 100
    return mae, rmse, mape.item()


In [37]:
def visualize_predictions(pred, target, scaler, output_dir='plots', num_sensors=3):
    os.makedirs(output_dir, exist_ok=True)

    # pred and target shapes: [batch_size, num_nodes, out_timestep]
    # Select the first batch and transpose to [out_timestep, num_nodes]
    pred = pred[0].cpu().numpy().T  # shape: [12, 207]
    target = target[0].cpu().numpy().T  # shape: [12, 207]

    # Inverse transform the entire batch for all sensors
    pred = scaler.inverse_transform(pred).T  # shape: [207, 12]
    target = scaler.inverse_transform(target).T  # shape: [207, 12]

    # Plot
    plt.figure(figsize=(12, 8))
    for i in range(min(num_sensors, pred.shape[0])):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(pred[i], label='Predicted', marker='o')
        plt.plot(target[i], label='Actual', marker='x')
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction')
        plt.xlabel('Time Step (5-min intervals)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'prediction_plot.png'))
    plt.close()

In [38]:
def main():
    torch.cuda.empty_cache()
    node_values, edge_index, edge_weight, scaler = load_metr_la_data(data_dir)
    features, labels = create_data_samples(node_values)

    transform = AddRandomWalkPE(walk_length=20, attr_name='pe')
    data_list = []
    for i in range(features.shape[0]):
        x = features[i].T
        y = labels[i].T
        data = Data(x=x, y=y, edge_index=edge_index)
        data = transform(data)
        data_list.append(data)

    train_data = data_list[:int(0.7 * len(data_list))]
    val_data = data_list[int(0.7 * len(data_list)):int(0.9 * len(data_list))]
    test_data = data_list[int(0.9 * len(data_list)):]

    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32)
    test_loader = DataLoader(test_data, batch_size=32)

    model = GraphTransformer(in_channels=12, hidden_channels=64, out_channels=12, num_layers=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    criterion = nn.MSELoss()

    for epoch in range(20):
        model.train()
        total_train_loss = 0
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.pe, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        train_loss = total_train_loss / len(train_loader)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                out = model(data.x, data.edge_index, data.pe, data.batch)
                batch_size = data.batch.max().item() + 1
                num_nodes = out.shape[1]
                target = data.y.view(batch_size, num_nodes, -1)
                loss = criterion(out, target)
                total_val_loss += loss.item()

        val_loss = total_val_loss / len(val_loader)
        print(f"Epoch {epoch + 1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    model.eval()
    total_mae, total_rmse, total_mape = 0, 0, 0
    num_batches = 0
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.pe, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            mae, rmse, mape = compute_metrics(out, target)
            total_mae += mae
            total_rmse += rmse
            total_mape += mape
            num_batches += 1
            if num_batches == 1:
                visualize_predictions(out, target, scaler)

    print(f"Test MAE: {total_mae / num_batches:.4f}")
    print(f"Test RMSE: {total_rmse / num_batches:.4f}")
    print(f"Test MAPE: {total_mape / num_batches:.4f}%")

if __name__ == '__main__':
    main()


Original node_values shape: (34272, 207, 2)
Processed node_values shape: (207, 34272)
edge_index device: cuda:0, shape: torch.Size([2, 1722])
edge_weight device: cuda:0, shape: torch.Size([1722])
Epoch 1, Train Loss: 0.0394, Val Loss: 0.2245
Epoch 2, Train Loss: 0.0234, Val Loss: 0.2788
Epoch 3, Train Loss: 0.0230, Val Loss: 0.3436
Epoch 4, Train Loss: 0.0227, Val Loss: 0.3232
Epoch 5, Train Loss: 0.0227, Val Loss: 0.3015
Epoch 6, Train Loss: 0.0225, Val Loss: 0.2997
Epoch 7, Train Loss: 0.0222, Val Loss: 0.3351
Epoch 8, Train Loss: 0.0222, Val Loss: 0.2095
Epoch 9, Train Loss: 0.0220, Val Loss: 0.2174
Epoch 10, Train Loss: 0.0219, Val Loss: 0.2215
Epoch 11, Train Loss: 0.0218, Val Loss: 0.2336
Epoch 12, Train Loss: 0.0217, Val Loss: 0.1588
Epoch 13, Train Loss: 0.0216, Val Loss: 0.1771
Epoch 14, Train Loss: 0.0215, Val Loss: 0.2530
Epoch 15, Train Loss: 0.0214, Val Loss: 0.2249
Epoch 16, Train Loss: 0.0213, Val Loss: 0.2187
Epoch 17, Train Loss: 0.0212, Val Loss: 0.2198
Epoch 18, Trai